In [1]:
import os
import time
import yaml
from ultralytics import YOLO, RTDETR
import torch
import shutil
import cv2
from ultralytics.models.yolo.classify import ClassificationTrainer

In [ ]:
# MODEL 1: YOLOv11-Medium (Single-Stage Convolutional Detector)
print("--- [PHASE 1] Launching YOLOv11-Medium Training ---")

# Initialize the Medium variant weights natively
model_yolo = YOLO("yolo11m.pt")

results_yolo = model_yolo.train(
    data="dataset.yaml",            
    epochs=30,                      
    imgsz=640,                      
    batch=16,                       
    device=0,                        
    amp=True,                        
    
    perspective=0.001,              # Random perspective matrix to simulate steep overhead camera pitch
    scale=0.5,                       # Forceful mosaic downscaling to train multi-distance scale anchors
    degrees=10.0,                    # Gentle rotational variance to handle diagonal entry approaches
    mosaic=1.0,
    mixup=0.15,
    fliplr=0.5,
    
    project="Safety_Pipeline_Runs",
    name="YOLOv11m_Baseline"
)

# Forcefully flush lingering feature map allocations from the GPU VRAM buffer
del model_yolo
torch.cuda.empty_cache()
print("Phase 1 Complete. Hardware memory cache flushed successfully.\n")

--- [PHASE 1] Launching YOLOv11-Medium Training ---
New https://pypi.org/project/ultralytics/8.4.51 available 😃 Update with 'pip install -U ultralytics'


/home/grimlock/miniconda3/envs/default/lib/python3.13/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Ultralytics 8.4.50 🚀 Python-3.13.5 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 7806MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=YOLOv11m_Baseline-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=

In [4]:
# MODEL 2: RT-DETR (Transformer-Based Attention Detector)
print("--- [PHASE 2] Launching RT-DETR Training ---")

# We use the natively supported baseline weights. 
model_rtdetr = RTDETR("rtdetr-l.pt")

results_rtdetr = model_rtdetr.train(
    data="dataset.yaml",
    epochs=30,                       
    imgsz=640,
    
    batch=3,                         # Loads only 3 images into VRAM at a time 
    nbs=16,                          # Accumulates gradients across 6 steps to maintain mathematical stability
    
    device=0,
    amp=False,                        
    
    perspective=0.001,              # Random perspective matrix to simulate steep overhead camera pitch
    scale=0.5,                       # Forceful mosaic downscaling to train multi-distance scale anchors
    degrees=10.0,                    # Gentle rotational variance to handle diagonal entry approaches
    mosaic=1.0,
    mixup=0.15,
    fliplr=0.5,
    
    project="Safety_Pipeline_Runs",
    name="RTDETR_Baseline_Accumulated"
)

# Forcefully flush lingering feature map allocations from the GPU VRAM buffer
del model_rtdetr
torch.cuda.empty_cache()
print("Phase 2 Complete. Hardware memory cache flushed successfully.\n")

--- [PHASE 2] Launching RT-DETR Training ---
New https://pypi.org/project/ultralytics/8.4.51 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.50 🚀 Python-3.13.5 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 7806MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=3, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=rtdetr-l.pt, 

/home/grimlock/miniconda3/envs/default/lib/python3.13/site-packages/torch/autograd/graph.py:841: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:148.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/30       4.5G     0.6589      1.659     0.3653          5        640: 100% ━━━━━━━━━━━━ 1382/1382 3.8it/s 6:07<0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 173/173 6.8it/s 25.3s0.1ss
                   all       1036       2719      0.544      0.392      0.403      0.223

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       2/30      4.43G     0.5121     0.7735       0.18         17        640: 0% ──────────── 0/1382  0.3s

/home/grimlock/miniconda3/envs/default/lib/python3.13/site-packages/torch/autograd/graph.py:841: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:148.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/30      4.43G     0.6268     0.8708     0.3401          7        640: 100% ━━━━━━━━━━━━ 1382/1382 3.9it/s 5:53<0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 173/173 6.8it/s 25.3s0.1ss
                   all       1036       2719      0.455      0.452      0.406      0.237

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       3/30       4.4G     0.6171     0.9215     0.2823         16        640: 0% ──────────── 0/1382  0.3s

...

      28/30      4.45G     0.3294     0.5482     0.2032          3        640: 100% ━━━━━━━━━━━━ 1382/1382 3.9it/s 5:59<0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 173/173 6.8it/s 25.4s0.1ss
                   all       1036       2719      0.807      0.787       0.86       0.45

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      29/30      4.41G     0.3989     0.4646     0.1171          5        640: 0% ──────────── 0/1382  0.3s

/home/grimlock/miniconda3/envs/default/lib/python3.13/site-packages/torch/autograd/graph.py:841: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:148.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      29/30      4.43G     0.3253     0.5306      0.197          3        640: 100% ━━━━━━━━━━━━ 1382/1382 3.8it/s 5:60<0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 173/173 6.7it/s 25.8s0.1ss
                   all       1036       2719      0.811       0.81      0.873       0.46

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      30/30       4.4G     0.2968     0.3397      0.491          3        640: 0% ──────────── 0/1382  0.3s

/home/grimlock/miniconda3/envs/default/lib/python3.13/site-packages/torch/autograd/graph.py:841: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:148.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      30/30      4.45G     0.3206     0.5237     0.1983         18        640: 100% ━━━━━━━━━━━━ 1382/1382 3.8it/s 6:02<0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 173/173 6.7it/s 25.8s0.1ss
                   all       1036       2719      0.805      0.801       0.87       0.46

30 epochs completed in 3.191 hours.
Optimizer stripped from /home/grimlock/Downloads/Project/Models/runs/detect/Safety_Pipeline_Runs/RTDETR_Baseline_Accumulated-3/weights/last.pt, 66.2MB
Phase 2 Complete. Hardware memory cache flushed successfully.



In [ ]:
def generate_classification_dataset(yolo_images_dir, yolo_labels_dir, output_root):
    # The class mapping defined in dataset.yaml
    classes = {0: "Helmet", 1: "Pagdi", 2: "No-Helmet"}
    
    # Create the output folders automatically
    for class_name in classes.values():
        os.makedirs(os.path.join(output_root, class_name), exist_ok=True)
        
    image_files = [f for f in os.listdir(yolo_images_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
    crop_counter = 0
    
    for img_file in image_files:
        img_path = os.path.join(yolo_images_dir, img_file)
        label_path = os.path.join(yolo_labels_dir, img_file.rsplit('.', 1)[0] + '.txt')
        
        if not os.path.exists(label_path):
            continue # Skip images with no labeled heads
            
        # Load the full street frame
        img = cv2.imread(img_path)
        h, w, _ = img.shape
        
        with open(label_path, 'r') as file:
            lines = file.readlines()
            
        for line in lines:
            parts = line.strip().split()
            class_id = int(float((parts[0])))
            
            # YOLO coordinates are normalized (0.0 to 1.0). Convert them back to exact pixels.
            x_center, y_center = float(parts[1]) * w, float(parts[2]) * h
            box_width, box_height = float(parts[3]) * w, float(parts[4]) * h
            
            x1 = int(x_center - (box_width / 2))
            y1 = int(y_center - (box_height / 2))
            x2 = int(x_center + (box_width / 2))
            y2 = int(y_center + (box_height / 2))
            
            # Ensure coordinates don't fall off the edge of the image
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w, x2), min(h, y2)
            
            # Slice the image array to create the tight crop
            crop = img[y1:y2, x1:x2]
            
            # Skip invalid, infinitesimally small crops
            if crop.shape[0] == 0 or crop.shape[1] == 0:
                continue
                
            # Save the crop directly into the correct class folder
            class_name = classes[class_id]
            save_path = os.path.join(output_root, class_name, f"crop_{crop_counter}.jpg")
            cv2.imwrite(save_path, crop)
            crop_counter += 1
            
    print(f"Successfully generated {crop_counter} tightly cropped images into {output_root}!")

generate_classification_dataset("Datasets/Detection-Dataset/images/train", "Datasets/Detection-Dataset/labels/train", "Datasets/Head-Crops-Dataset/train")

Successfully generated 11015 tightly cropped images into Datasets/Head-Crops-Dataset/train!


In [ ]:
# MODEL 3: ResNet-50 (Decoupled Image Classifier Head)
print("--- [PHASE 3] Launching ResNet-50 Attribute Classifier ---")

model_resnet = YOLO("yolov8-cls-resnet50.yaml") 

results_resnet = model_resnet.train(
    data="/home/grimlock/Downloads/Project/Models/Datasets/Head-Crops-Dataset",       
    epochs=30,
    patience=10,
    imgsz=224,                       
    batch=64,                        
    device=0,
    amp=False,                        
    
    # Custom pixel-level noise for cropped patches
    degrees=5.0,
    fliplr=0.5,
    
    project="Safety_Pipeline_Runs",
    name="ResNet50_Classifier"
)

# Forcefully flush the VRAM buffer
del model_resnet
torch.cuda.empty_cache()
print("Phase 3 Complete. Hardware memory cache flushed successfully.\n")

--- [PHASE 3] Launching ResNet-50 Attribute Classifier ---
WARNING ⚠️ no model scale passed. Assuming scale='n'.
YOLOv8-cls-resnet50 summary: 145 layers, 27,413,032 parameters, 27,413,032 gradients, 70.6 GFLOPs
New https://pypi.org/project/ultralytics/8.4.51 available 😃 Update with 'pip install -U ultralytics'


/home/grimlock/miniconda3/envs/default/lib/python3.13/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Ultralytics 8.4.50 🚀 Python-3.13.5 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 7806MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/grimlock/Downloads/Project/Models/Datasets/Head-Crops-Dataset, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8-cls-resnet50.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=ResNet50_Classifier-3, nbs=64

In [3]:
def evaluate_pipeline():
    print("--- Initiating Pipeline Metrics Evaluation ---")

    # Define the paths to the best weights generated by your training runs
    models_to_evaluate = {
        "YOLOv11-Medium (Detection)": "runs/detect/Safety_Pipeline_Runs/YOLOv11m_Baseline/weights/best.pt",
        "RT-DETR-Baseline (Detection)": "runs/detect/Safety_Pipeline_Runs/RTDETR_Baseline_Accumulated/weights/best.pt",
        "Resnet-50 (Classification)": "runs/classify/Safety_Pipeline_Runs/ResNet50_Classifier/weights/best.pt"
    }

    for model_name, weight_path in models_to_evaluate.items():
        if not os.path.exists(weight_path):
            print(f"[WARNING] Weights not found for {model_name} at {weight_path}. Skipping...\n")
            continue
            
        print(f"--- Evaluating {model_name} ---")
        
        # Load the correct architecture wrapper
        is_classifier = "Classification" in model_name
        model = RTDETR(weight_path) if "RT-DETR" in model_name else YOLO(weight_path)
        
        # Extract Hardware & Complexity Metrics
        # model.info() natively calculates parameters and GFLOPs
        model_info = model.info()
        
        # Run rigorous validation on the holdout set
        metrics = model.val(half=True, split='val')
        
        print("\n[ HARDWARE & EFFICIENCY ]")
        print(f"  -> Parameters: {model_info[0]:,} ")
        print(f"  -> GFLOPs:     {model_info[1]:.1f}")
        # Inference speed is measured in milliseconds per frame (ms/frame)
        print(f"  -> Latency:    {metrics.speed['inference']:.2f} ms/frame")
        
        print("\n[ PERFORMANCE METRICS ]")
        if is_classifier:
            # Classification-specific metrics
            print(f"  -> Top-1 Accuracy: {metrics.top1:.4f}")
            print(f"  -> Top-5 Accuracy: {metrics.top5:.4f}")

        else:
            # Detection-specific metrics
            print(f"  -> mAP@50:     {metrics.box.map50:.4f}")
            print(f"  -> mAP@50-95:  {metrics.box.map:.4f}")
            
            # Extracting class-specific recall requires pulling the arrays
            if len(metrics.box.ap_class_index) >= 3:
                # Find the index for the 'No-Helmet' class
                no_helmet_idx = list(metrics.box.ap_class_index).index(2) if 2 in metrics.box.ap_class_index else -1
                if no_helmet_idx != -1:
                    no_helmet_recall = metrics.box.r[no_helmet_idx]
                    print(f"  -> No-Helmet Recall: {no_helmet_recall:.4f}")
        
        print(f"\nVisual plots saved to: {metrics.save_dir}")
        print("-------------------------------------------------------\n")

evaluate_pipeline()

--- Initiating Pipeline Metrics Evaluation ---
--- Evaluating YOLOv11-Medium (Detection) ---
YOLO11m summary: 232 layers, 20,055,321 parameters, 0 gradients, 68.2 GFLOPs
Ultralytics 8.4.50 🚀 Python-3.13.5 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 7806MiB)
YOLO11m summary (fused): 126 layers, 20,032,345 parameters, 0 gradients, 67.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4913.8±4232.0 MB/s, size: 127.8 KB)
val: Scanning /home/grimlock/Downloads/Project/Models/Datasets/Detection-Dataset/labels/val.cache... 1036 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1036/1036 620.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 65/65 8.7it/s 7.5s<0.1s
                   all       1036       2719      0.878       0.86       0.93      0.753
                Helmet        349        960      0.821       0.86      0.909      0.723
                 Pagdi        300        417      0.915    